# Pruebas pequeñas de los estadisticos 

## 1) t de Welch

In [1]:
import numpy as np
from scipy import stats

x = np.array([1.2, 2.0, 2.4, 1.8])
y = np.array([2.1, 2.5, 3.0, 2.7, 2.8])

# SciPy
res = stats.ttest_ind(x, y, equal_var=False)
print("SciPy t:", res.statistic)
print("SciPy p:", res.pvalue)

# Manual Welch
mx, my = x.mean(), y.mean()
sx2, sy2 = x.var(ddof=1), y.var(ddof=1)
m, n = len(x), len(y)

t_manual = (mx - my) / np.sqrt(sx2/m + sy2/n)
df = (sx2/m + sy2/n)**2 / ((sx2/m)**2/(m-1) + (sy2/n)**2/(n-1))
p_manual = 2 * stats.t.sf(abs(t_manual), df)

print("Manual t:", t_manual)
print("Manual p:", p_manual)

SciPy t: -2.627206096649522
SciPy p: 0.0455535497945293
Manual t: -2.627206096649522
Manual p: 0.04555354979452928


## 2) Wilcoxon–Mann–Whitney

In [2]:
import numpy as np
from scipy import stats

x = np.array([1, 3, 5])
y = np.array([2, 4])

# SciPy
res = stats.mannwhitneyu(x, y, alternative='two-sided')
print("SciPy U:", res.statistic)
print("SciPy p:", res.pvalue)

# Conteo manual de U = sum 1(x_i < y_j)
u_manual = sum(int(xi < yj) for xi in x for yj in y)
print("Manual U:", u_manual)

SciPy U: 3.0
SciPy p: 1.0
Manual U: 3


## 3) Hodges–Lehmann

In [3]:
import numpy as np
from scipy import stats

ALPHA = 0.05
x = np.array([1, 2, 3])
y = np.array([3, 4])

m, n = len(x), len(y)
mn = m * n

diffs = (y[:, None] - x).flatten()
print("Diferencias:", diffs)
print("Diferencias ordenadas:", np.sort(diffs))

mean_u = mn / 2
std_u = np.sqrt(mn * (m + n + 1) / 12)
z_alpha = stats.norm.ppf(1 - ALPHA / 2)

lower_idx = max(0, int(np.round(mean_u - z_alpha * std_u)) - 1)
upper_idx = min(mn - 1, int(np.round(mean_u + z_alpha * std_u)) - 1)

partitioned = np.partition(diffs, [lower_idx, upper_idx])
ci_lower = partitioned[lower_idx]
ci_upper = partitioned[upper_idx]

print("IC aproximado HL:", (ci_lower, ci_upper))
print("Rechaza H0:", not (ci_lower <= 0 <= ci_upper))

Diferencias: [2 1 0 3 2 1]
Diferencias ordenadas: [0 1 1 2 2 3]
IC aproximado HL: (np.int64(0), np.int64(3))
Rechaza H0: False


## 4) Permutación, con media/mediana/trim

In [4]:
import numpy as np
from scipy import stats
from itertools import combinations

x = np.array([1, 2, 3])
y = np.array([4, 5])

def diff_medias(a, b):
    return np.mean(a) - np.mean(b)

def diff_medianas(a, b):
    return np.median(a) - np.median(b)

def diff_trim(a, b):
    return stats.trim_mean(a, 0.1) - stats.trim_mean(b, 0.1)

for name, stat_func in [("media", diff_medias), ("mediana", diff_medianas), ("trim", diff_trim)]:
    res = stats.permutation_test(
        (x, y), stat_func,
        permutation_type='independent',
        n_resamples=1000,
        alternative='two-sided'
    )
    print(name, "SciPy stat:", res.statistic, "p:", res.pvalue)

# Enumeración exacta para diferencia de medias
pool = np.concatenate([x, y])
m = len(x)
obs = diff_medias(x, y)

perm_stats = []
for idx in combinations(range(len(pool)), m):
    idx = set(idx)
    x_perm = np.array([pool[i] for i in range(len(pool)) if i in idx])
    y_perm = np.array([pool[i] for i in range(len(pool)) if i not in idx])
    perm_stats.append(diff_medias(x_perm, y_perm))

perm_stats = np.array(perm_stats)
p_exact = np.mean(np.abs(perm_stats) >= abs(obs))
print("Permutación exacta (media) p:", p_exact)

media SciPy stat: -2.5 p: 0.2
mediana SciPy stat: -2.5 p: 0.2
trim SciPy stat: -2.5 p: 0.2
Permutación exacta (media) p: 0.2
